# <font color="#C0392B"> Examen Big Data – Spark ML: predicción de deserción y éxito académico </font>

**Profesor:** Alfredo Diaz Claro  
**Asignatura / Módulo:** Big Data

**Notebook comentado de la solución del ejercicio práctico (Parte 2)**, con la relación explícita con las 20 preguntas del examen teórico-práctico (Parte 1, "completar").

### Cómo leer este notebook
- Cada celda de código va **precedida por una celda de texto** que explica qué hace, por qué se hace así y qué tarea del enunciado cubre.
- En el código, las líneas marcadas con `# Teoría P#` son el lugar exacto donde se ve la respuesta de la pregunta # del examen teórico.
- Las celdas están numeradas (A.1, A.2, …) para ubicarlas fácilmente.

### Mapa rápido: pregunta teórica → dónde encontrar la respuesta
| P# | Pregunta (resumen) | Respuesta | Dónde verla en el notebook |
|---|---|---|---|
| 1 | Proceso que coordina la app, crea el SparkContext y reparte tareas a los executors | **Driver** (Driver Program) | A.2 (la `SparkSession` se crea en el driver) y D.3 |
| 2 | Las transformaciones son de evaluación ___ | **perezosa (lazy)** | A.3 y A.5 (transformaciones) frente a A.4 (acciones `count()`, `show()`) |
| 3 | `SparkSession.___.appName(...)` | **builder** | A.2 |
| 4 | Parámetro que detecta el tipo de dato de cada columna | **inferSchema** | A.2 |
| 5 | `df.na.___()` elimina filas con nulos | **drop** | A.3 |
| 6 | Parámetro de `StringIndexer` con el nombre de la columna de salida | **outputCol** | A.6 (y B.2) |
| 7 | Transformador que convierte índices en vectores binarios (dummy) | **OneHotEncoder** | B.2 |
| 8 | Clase que une varias columnas en una de características | **VectorAssembler** | B.2 |
| 9 | Las variables predictoras deben estar en una única columna de tipo ___ | **Vector** (`VectorUDT`) | B.2 (columna `features_raw`) |
| 10 | Clase que estandariza a desviación estándar unitaria | **StandardScaler** | B.2 |
| 11 | El ___ implementa `fit()` y produce un modelo; el Transformer implementa `transform()` | **Estimator** | A.6 (`StringIndexer(...).fit`), B.3 (`rf`, `lr`, `dt`) y B.6 (`cv.fit`) |
| 12 | Clase que encadena preprocesamiento y modelo | **Pipeline** | B.6 |
| 13 | Método que divide el DataFrame en train y test | **randomSplit** | A.6 |
| 14 | Hiperparámetro con la cantidad de árboles del bosque | **numTrees** | B.3 y B.4 |
| 15 | Valor de `family` para regresión logística multiclase | **multinomial** | B.3 |
| 16 | Método que construye la lista final de combinaciones | **build** | B.4 |
| 17 | Parámetro del número de particiones del `CrossValidator` | **numFolds** | B.6 |
| 18 | Evaluador para clasificación con más de dos clases | **MulticlassClassificationEvaluator** | B.5 y C.2 |
| 19 | Atributo que devuelve el mejor modelo de la validación cruzada | **bestModel** | C.1 |
| 20 | Método que guarda el modelo en disco | **save** (`write().overwrite().save(...)`) | C.4 |

> Conceptos de apoyo: **Estimator** → `fit()` devuelve un **Transformer/Modelo**. Ejemplos: `StringIndexer`, `OneHotEncoder`, `StandardScaler`, `RandomForestClassifier`, `Pipeline`, `CrossValidator`. **Transformer** → `transform()`. Ejemplos: `VectorAssembler`, `PipelineModel`, `StringIndexerModel`.

Marital status (Estado civil): El estado civil del estudiante. (Categórica)

Application mode (Modo de postulación): El método de solicitud utilizado por el estudiante. (Categórica)

Application order (Orden de postulación): El orden en el que el estudiante realizó la solicitud. (Numérica)

Course (Programa / Carrera): El programa académico o curso que toma el estudiante. (Categórica)

Daytime/evening attendance (Jornada diurna/nocturna): Indica si el estudiante asiste a clases durante el día o en la noche. (Categórica)

Previous qualification (Formación previa): La titulación o nivel académico obtenido por el estudiante antes de ingresar a la educación superior. (Categórica)

Nacionality (Nacionalidad): La nacionalidad del estudiante. (Categórica)

Mother's qualification (Nivel educativo de la madre): La titulación o nivel académico de la madre del estudiante. (Categórica)

Father's qualification (Nivel educativo del padre): La titulación o nivel académico del padre del estudiante. (Categórica)

Mother's occupation (Ocupación de la madre): La ocupación de la madre del estudiante. (Categórica)

Father's occupation (Ocupación del padre): La ocupación del padre del estudiante. (Categórica)

Displaced (Desplazado): Indica si el estudiante es una persona en situación de desplazamiento. (Categórica)

Educational special needs (Necesidades educativas especiales): Indica si el estudiante tiene alguna necesidad educativa especial. (Categórica)

Debtor (Deudor): Indica si el estudiante tiene deudas pendientes. (Categórica)

Tuition fees up to date (Matrícula al día): Indica si los pagos de matrícula del estudiante están al día. (Categórica)

Gender (Género): El género del estudiante. (Categórica)

Scholarship holder (Becado): Indica si el estudiante cuenta con una beca. (Categórica)

Age at enrollment (Edad al matricularse): La edad del estudiante al momento de la inscripción. (Numérica)

International (Internacional): Indica si el estudiante es internacional. (Categórica)

Curricular units 1st sem (credited) (Unidades curriculares 1.er sem. - acreditadas/homologadas): El número de unidades curriculares (asignaturas) acreditadas por el estudiante en el primer semestre. (Numérica)

Curricular units 1st sem (enrolled) (Unidades curriculares 1.er sem. - matriculadas): El número de unidades curriculares inscritas por el estudiante en el primer semestre. (Numérica)

Curricular units 1st sem (evaluations) (Unidades curriculares 1.er sem. - evaluadas): El número de unidades curriculares evaluadas al estudiante en el primer semestre. (Numérica)

Curricular units 1st sem (approved) (Unidades curriculares 1.er sem. - aprobadas): El número de unidades curriculares aprobadas por el estudiante en el primer semestre. (Numérica)

## A. Carga, limpieza, ingeniería de características y partición
*(Cubre las tareas 1, 2 y 3 del enunciado: carga y exploración, preprocesamiento e ingeniería de características.)*

### A.1 Importaciones
Se traen todas las clases que se usarán después, agrupadas por función:
- `SparkSession` y `functions as F`: crear la sesión y manipular columnas (`F.when`, `F.col`).
- `pyspark.ml.feature`: **transformadores de preprocesamiento** (`StringIndexer`, `OneHotEncoder`, `VectorAssembler`, `StandardScaler`).
- `pyspark.ml.classification`: los **tres modelos** pedidos (regresión logística, bosque aleatorio y árbol de decisión).
- `ParamGridBuilder` y `CrossValidator`: **hiperparametrización**.
- `MulticlassClassificationEvaluator`: **métricas** para tres clases.
- `Pipeline`: encadena preprocesamiento + modelo.

**Teoría:** aquí aparecen los *nombres* de las respuestas de P7, P8, P10, P12 y P18 (se importan, y se usan más adelante en la sección B).

In [ ]:
import re
from pyspark.sql import SparkSession, functions as F
from pyspark.ml import Pipeline                                  #  Teoría P12 (clase Pipeline)
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler   #  P7, P8, P10
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, DecisionTreeClassifier
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import MulticlassClassificationEvaluator   #  Teoría P18

### A.2 Creación de la SparkSession y carga del CSV  *(Tarea 1)*
- `SparkSession.builder.appName(...).getOrCreate()` es el **punto de entrada** a Spark. Se crea en el proceso **Driver**, que a su vez crea el `SparkContext` y reparte el trabajo entre los *executors*.
- `spark.read.csv(..., header=True, inferSchema=True, sep=";")`:
  - `header=True`: la primera fila contiene los nombres de columna.
  - `inferSchema=True`: Spark recorre los datos y **detecta automáticamente el tipo** (entero, double, string) de cada columna; si no se activa, todo se lee como `string`.
  - `sep=";"`: este dataset de Kaggle usa punto y coma (por eso el comentario "verificar el separador").

 **Teoría:** **P1** (Driver), **P3** (`builder`) y **P4** (`inferSchema`).

In [ ]:
spark = (SparkSession.builder
         .appName("dropout_sparkml")
         .config("spark.driver.memory", "4g")
         .config("spark.sql.constraintPropagation.enabled", "false")   # evita la explosión del optimizador
         .getOrCreate())

df = spark.read.csv("dataset.csv", header=True, inferSchema=True, sep=",")

In [ ]:
#Mostrar algunas muestras de df
df.show(5)

+--------------+----------------+-----------------+------+--------------------------+----------------------+-----------+----------------------+----------------------+-------------------+-------------------+---------+-------------------------+------+-----------------------+------+------------------+-----------------+-------------+-----------------------------------+-----------------------------------+--------------------------------------+-----------------------------------+--------------------------------+----------------------------------------------+-----------------------------------+-----------------------------------+--------------------------------------+-----------------------------------+--------------------------------+----------------------------------------------+-----------------+--------------+-----+--------+
|Marital status|Application mode|Application order|Course|Daytime/evening attendance|Previous qualification|Nacionality|Mother's qualification|Father's qualification|

### A.3 Normalización de nombres y limpieza  *(Tarea 2)*
1. **Normalizar nombres de columnas**: `re.sub(r"\W+", "_", ...)` cambia espacios, paréntesis, tildes y símbolos por guiones bajos, y se pasa a minúsculas. Así `"Curricular units 1st sem (approved)"` queda `curricular_units_1st_sem_approved`, que es seguro para usar en Spark SQL y en el `VectorAssembler`.
2. **Duplicados y nulos**: `dropDuplicates()` elimina filas repetidas y `na.drop()` elimina las filas con valores nulos.

**Teoría:**
- **P5**: `df.na.drop()` es exactamente la respuesta (**drop**).
- **P2 (evaluación perezosa)**: `toDF`, `dropDuplicates` y `na.drop` son **transformaciones**; Spark solo construye el plan y *no ejecuta nada* hasta que aparece una acción (en la celda A.4: `count()` y `show()`).

In [ ]:
df = df.toDF(*[re.sub(r"\W+", "_", c.strip().lower()).strip("_") for c in df.columns])
df = df.dropDuplicates().na.drop()        # 🔎 Teoría P5 (na.drop) | P2: son transformaciones perezosas, aún no se ejecutan

### A.4 Exploración: esquema, dimensiones y distribución de clases  *(Tareas 1 y 2)*
- `printSchema()` muestra nombres y **tipos** de dato (comprueba el efecto de `inferSchema`).
- `df.count()` y `len(df.columns)` dan el **número de filas y de columnas**.
- `groupBy("target").count().show()` muestra la **distribución de las clases** (Dropout, Enrolled, Graduate). Aquí se evidencia el **desbalance**: Graduate es la clase mayoritaria y Enrolled la minoritaria.

 **Teoría (P2):** `count()` y `show()` son **acciones**: es en este momento cuando Spark ejecuta todas las transformaciones pendientes de A.2 y A.3. Es el ejemplo literal que menciona el enunciado de la pregunta 2.

 **Pendiente según el examen/rúbrica:** el enunciado pide además *decidir y justificar una estrategia para el desbalance*. Opciones típicas:

 (a) usar **F1 ponderado** como métrica de selección (ya se hace en B.5),

 (b) agregar una columna de **pesos por clase** (`weightCol`), o

 (c) sobremuestreo/submuestreo. Conviene escribir aquí una celda de texto con la estrategia elegida y su justificación.

In [ ]:
df.printSchema(); print(df.count(), len(df.columns))      # 🔎 Teoría P2: count() es una ACCIÓN
df.groupBy("target").count().show()                                      # 🔎 Teoría P2: show() es una ACCIÓN | distribución de clases

root
 |-- marital_status: integer (nullable = true)
 |-- application_mode: integer (nullable = true)
 |-- application_order: integer (nullable = true)
 |-- course: integer (nullable = true)
 |-- daytime_evening_attendance: integer (nullable = true)
 |-- previous_qualification: integer (nullable = true)
 |-- nacionality: integer (nullable = true)
 |-- mother_s_qualification: integer (nullable = true)
 |-- father_s_qualification: integer (nullable = true)
 |-- mother_s_occupation: integer (nullable = true)
 |-- father_s_occupation: integer (nullable = true)
 |-- displaced: integer (nullable = true)
 |-- educational_special_needs: integer (nullable = true)
 |-- debtor: integer (nullable = true)
 |-- tuition_fees_up_to_date: integer (nullable = true)
 |-- gender: integer (nullable = true)
 |-- scholarship_holder: integer (nullable = true)
 |-- age_at_enrollment: integer (nullable = true)
 |-- international: integer (nullable = true)
 |-- curricular_units_1st_sem_credited: integer (nullable

### A.5 Ingeniería de características  *(Tarea 3)*
Se crean **3 variables derivadas con sentido académico** (el examen exige al menos 3):

| Variable | Fórmula | Interpretación |
|---|---|---|
| `tasa_aprob_s1` | aprobadas 1.er sem. / inscritas 1.er sem. | Qué proporción de las materias del primer semestre aprobó el estudiante |
| `tasa_aprob_s2` | aprobadas 2.º sem. / inscritas 2.º sem. | Lo mismo para el segundo semestre |
| `delta_nota` | nota 2.º sem. − nota 1.er sem. | Tendencia del rendimiento: positiva = mejora, negativa = empeora |

- `F.when(inscritas > 0, aprobadas/inscritas).otherwise(0.0)` evita la **división por cero** cuando el estudiante no se inscribió en materias.
- `withColumn` es una **transformación** (perezosa): agrega una columna nueva sin modificar el DataFrame original.

 **Teoría (P2):** de nuevo transformaciones perezosas; nada se calcula hasta la siguiente acción.

In [ ]:
# Ingeniería de características (los nombres exactos dependen de la versión del CSV)
df = (df
  .withColumn("tasa_aprob_s1", F.when(F.col("curricular_units_1st_sem_enrolled") > 0,
        F.col("curricular_units_1st_sem_approved") / F.col("curricular_units_1st_sem_enrolled")).otherwise(0.0))
  .withColumn("tasa_aprob_s2", F.when(F.col("curricular_units_2nd_sem_enrolled") > 0,
        F.col("curricular_units_2nd_sem_approved") / F.col("curricular_units_2nd_sem_enrolled")).otherwise(0.0))
  .withColumn("delta_nota", F.col("curricular_units_2nd_sem_grade") - F.col("curricular_units_1st_sem_grade")))   # 🔎 Teoría P2 (withColumn = transformación)

In [ ]:
df = df.cache()
print(df.count())   # fuerza la ejecución y deja el resultado en memoria

4424


### A.6 Indexación de la etiqueta y partición entrenamiento/prueba  *(Tareas 2 y 4)*
- **`StringIndexer(inputCol="target", outputCol="label")`** convierte el texto de la clase (`Graduate`, `Dropout`, `Enrolled`) en un número (0, 1, 2) ordenado por frecuencia. Spark ML exige etiquetas numéricas.
  - `outputCol` es el parámetro que define el **nombre de la columna de salida**.
  - `StringIndexer` es un **Estimator**: se llama `.fit(df)` y devuelve un **StringIndexerModel** (un Transformer) que aprende el mapa texto → número.
- `label_model.labels` guarda ese orden. **Es clave para la app**: la posición en la lista corresponde al índice que predice el modelo, por eso la app (sección D) usa `LABELS = ["Graduate", "Dropout", "Enrolled"]`.
- La etiqueta se indexa **fuera del pipeline** a propósito: así el pipeline guardado no necesita la columna `target` al predecir con los datos de un estudiante nuevo.
- **`df.randomSplit([0.8, 0.2], seed=42)`** divide en 80 % entrenamiento y 20 % prueba, con **semilla fija 42** para que sea reproducible.

 **Teoría:** **P6** (`outputCol`), **P11** (Estimator con `fit()` → modelo, luego `transform()`) y **P13** (`randomSplit`).

In [ ]:
# Etiqueta indexada FUERA del pipeline para poder predecir en la app sin la columna target
label_model = StringIndexer(inputCol="target", outputCol="label").fit(df)   # Teoría P6 (outputCol) | P11: StringIndexer es un Estimator, .fit() devuelve un modelo
labels = label_model.labels                  # orden por frecuencia, p. ej. ['Graduate', 'Dropout', 'Enrolled']
df = label_model.transform(df)               #  Teoría P11: el modelo ajustado es un Transformer y usa transform()
train, test = df.randomSplit([0.8, 0.2], seed=42)   #  Teoría P13 (randomSplit) | 80/20 con semilla fija 42

In [ ]:
# 1) Menos particiones: el dataset es pequeño
spark.conf.set("spark.sql.shuffle.partitions", "4")

# 2) Cortar el historial de transformaciones: convertir a pandas y volver a Spark
train = spark.createDataFrame(train.toPandas()).repartition(4).cache()
test  = spark.createDataFrame(test.toPandas()).repartition(4).cache()
print(train.count(), test.count())   # fuerza el cache

3542 882


## B. Pipelines, hiperparametrización y validación cruzada
*(Cubre las tareas 3 (codificación, ensamblado y escalado), 4 (modelos con Pipeline) y 5 (hiperparametrización).)*

### B.1 Separación de columnas categóricas y numéricas
- `cat_cols`: variables categóricas codificadas como números pero que **no tienen orden** (estado civil, curso, modo de aplicación, etc.). Si se usaran tal cual, el modelo creería que el curso 9500 es "mayor" que el 33.
- `num_cols`: todas las demás (incluidas las 3 variables derivadas), excluyendo `target` y `label`.

In [ ]:
cat_cols = ["marital_status", "application_mode", "course", "previous_qualification", "gender", "scholarship_holder"]
num_cols = [c for c in df.columns if c not in cat_cols + ["target", "label"]]

### B.2 Etapas de preprocesamiento del Pipeline  *(Tarea 3)*
Esta celda define, en orden, los pasos que transforman los datos crudos en el vector que entiende el modelo:

1. **`StringIndexer`** (uno por columna categórica): asigna un índice numérico a cada categoría. `handleInvalid="keep"` evita errores si en la app llega una categoría no vista durante el entrenamiento.
2. **`OneHotEncoder`**: convierte cada índice en un **vector binario (variables dummy)**, de modo que no se asuma un orden entre categorías.
3. **`VectorAssembler`**: **une todas las columnas** (numéricas + dummies) en **una sola columna** `features_raw`. Spark ML exige que todas las predictoras estén reunidas en una única columna de tipo **Vector**.
4. **`StandardScaler`**: **estandariza** las características a desviación estándar unitaria, útil sobre todo para la regresión logística (que es sensible a la escala).

Las cuatro etapas irán **dentro del Pipeline** (ver B.6), requisito de la rúbrica ("codificación, ensamblado y escalado dentro del pipeline").

**Teoría:** **P7** (`OneHotEncoder`), **P8** (`VectorAssembler`), **P9** (columna de tipo **Vector**) y **P10** (`StandardScaler`). **P11**: `StringIndexer`, `OneHotEncoder` y `StandardScaler` son Estimators; `VectorAssembler` es un Transformer (no necesita `fit`).

 **Pendiente según el examen/rúbrica:** el enunciado pide *justificar la selección de variables* (por ejemplo con `ChiSqSelector` o con la importancia de características del Random Forest). No está en este código; se puede agregar una celda tras B.6 usando `featureImportances` del mejor Random Forest.

In [ ]:
idx = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in cat_cols]   #  Teoría P6/P11: StringIndexer (Estimator)
ohe = OneHotEncoder(inputCols=[f"{c}_idx" for c in cat_cols], outputCols=[f"{c}_ohe" for c in cat_cols])   #  Teoría P7 (OneHotEncoder)
asm = VectorAssembler(inputCols=num_cols + [f"{c}_ohe" for c in cat_cols], outputCol="features_raw")   #  Teoría P8 (VectorAssembler) | P9: la salida es UNA columna tipo Vector
scaler = StandardScaler(inputCol="features_raw", outputCol="features")   #  Teoría P10 (StandardScaler)

### B.3 Los tres modelos de clasificación multiclase  *(Tarea 4)*
Se definen tres algoritmos distintos (el examen pide al menos dos o tres):

- **`RandomForestClassifier`**: conjunto de muchos árboles; robusto y maneja bien relaciones no lineales. Los `numTrees` árboles votan.
- **`LogisticRegression(family="multinomial")`**: modelo lineal. `family="multinomial"` es lo que permite **clasificación multiclase** (3 clases); con `"binomial"` solo serviría para 2.
- **`DecisionTreeClassifier`**: un solo árbol; simple e interpretable.

Todos usan `labelCol="label"` (etiqueta indexada) y `featuresCol="features"` (vector escalado). `seed=42` hace los resultados reproducibles.

 **Teoría:** **P14** (`numTrees` es el hiperparámetro de cantidad de árboles; aquí se ajusta en la grilla de B.4), **P15** (`family="multinomial"`) y **P11** (estos tres son **Estimators**: su `fit()` produce un modelo entrenado).

In [ ]:
rf = RandomForestClassifier(labelCol="label", featuresCol="features", seed=42)    # 🔎 Teoría P14: su hiperparámetro numTrees se define en la grilla (B.4) | P11: Estimator
lr = LogisticRegression(labelCol="label", featuresCol="features", family="multinomial", maxIter=100)   # 🔎 Teoría P15 (multinomial)
dt = DecisionTreeClassifier(labelCol="label", featuresCol="features", seed=42)

### B.4 Mallas de hiperparámetros  *(Tarea 5)*
`ParamGridBuilder` arma todas las combinaciones de hiperparámetros a probar. Cada modelo tiene **2 hiperparámetros** (mínimo exigido):

| Modelo | Hiperparámetros | Combinaciones |
|---|---|---|
| Random Forest | `numTrees` [50, 100] × `maxDepth` [5, 10] | 4 |
| Regresión logística | `regParam` [0, 0.01, 0.1] × `elasticNetParam` [0, 0.5] | 6 |
| Árbol de decisión | `maxDepth` [3, 5, 10] × `minInstancesPerNode` [1, 5] | 6 |

`.addGrid(parámetro, lista)` agrega un hiperparámetro y `.build()` **construye la lista final de combinaciones** (cada una es un `ParamMap`).

 **Teoría:** **P16** (`build()`) y **P14** (`rf.numTrees`).

In [ ]:
grids = {
  "rf": ParamGridBuilder().addGrid(rf.numTrees, [50, 100]).addGrid(rf.maxDepth, [5, 10]).build(),     #  Teoría P14 (numTrees) y P16 (build)
  "lr": ParamGridBuilder().addGrid(lr.regParam, [0.0, 0.01, 0.1]).addGrid(lr.elasticNetParam, [0.0, 0.5]).build(),
  "dt": ParamGridBuilder().addGrid(dt.maxDepth, [3, 5, 10]).addGrid(dt.minInstancesPerNode, [1, 5]).build(),
}

### B.5 Evaluador  *(Tareas 5 y 6)*
`MulticlassClassificationEvaluator` es el evaluador de Spark ML para **clasificación con más de dos clases**. Con `metricName="f1"` usa el **F1 ponderado**, que promedia el F1 de cada clase ponderado por su frecuencia. Es una buena métrica para este problema porque las clases están **desbalanceadas**, y la usa el `CrossValidator` para elegir los mejores hiperparámetros.

 **Teoría:** **P18** (`MulticlassClassificationEvaluator`).

In [ ]:
ev = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")   # 🔎 Teoría P18

### B.6 Entrenamiento: Pipeline + CrossValidator para cada modelo  *(Tareas 4 y 5)*
Para cada uno de los tres modelos:
1. **`Pipeline(stages=idx + [ohe, asm, scaler, est])`** **encadena** el preprocesamiento y el modelo en un solo flujo. Así los mismos pasos se aplican de forma idéntica en entrenamiento, prueba y en la app.
2. **`CrossValidator`** prueba cada combinación de la malla con **`numFolds=3`** particiones (folds): entrena con 2 y valida con 1, rota, y promedia el F1. Escoge la mejor combinación y reentrena con todo `train`.
3. `cv.fit(train)` **entrena solo con el conjunto de entrenamiento**; luego `ev.evaluate(cv_model.transform(test))` mide el **F1 en el conjunto de prueba** (datos que el modelo no vio).
4. El resultado se guarda en el diccionario `resultados` como `(modelo_cv, f1_en_test)`.

 **Teoría:** **P12** (`Pipeline`), **P17** (`numFolds`) y **P11**: `Pipeline` y `CrossValidator` son **Estimators**: `fit()` produce `PipelineModel` / `CrossValidatorModel`, que son **Transformers** (se usa `transform()`).

In [ ]:
import time

resultados = {}
modelos_lista = [("rf", rf), ("lr", lr), ("dt", dt)]

for i, (name, est) in enumerate(modelos_lista, start=1):
    print(f"\n[{i}/{len(modelos_lista)}] Entrenando '{name}' "
          f"({len(grids[name])} combinaciones × 3 folds)...", flush=True)
    t0 = time.time()
    pipe = Pipeline(stages=idx + [ohe, asm, scaler, est])                                   #  Teoría P12 (Pipeline)
    cv = CrossValidator(estimator=pipe, estimatorParamMaps=grids[name], evaluator=ev, numFolds=3, seed=42)   #  Teoría P17 (numFolds)
    cv_model = cv.fit(train)                                                                #  Teoría P11: Estimator.fit() -> modelo entrenado
    resultados[name] = (cv_model, ev.evaluate(cv_model.transform(test)))                    #  Teoría P11: Transformer.transform() sobre el conjunto de prueba


[1/3] Entrenando 'rf' (4 combinaciones × 3 folds)...

[2/3] Entrenando 'lr' (6 combinaciones × 3 folds)...

[3/3] Entrenando 'dt' (6 combinaciones × 3 folds)...


## C. Evaluación, selección y guardado del mejor modelo
*(Cubre la tarea 6 del enunciado.)*

### C.1 Selección del mejor modelo
- `max(resultados, key=...)` escoge el modelo con **mayor F1 en el conjunto de prueba**.
- `resultados[best_name][0].bestModel` extrae, del `CrossValidatorModel`, el **mejor modelo encontrado por la validación cruzada**: es un `PipelineModel` completo (preprocesamiento + clasificador con los mejores hiperparámetros).
- `pred = best_model.transform(test)` genera las predicciones sobre `test`.

 **Teoría:** **P19** (`bestModel`).

 **Sugerencia para la rúbrica:** la rúbrica pide *comparar los modelos en una tabla* y *analizar los mejores parámetros*. Con el diccionario `resultados` se puede imprimir una tabla `modelo → F1` y, para cada modelo, los hiperparámetros ganadores con `bestModel.stages[-1].extractParamMap()`.

In [ ]:
best_name = max(resultados, key=lambda k: resultados[k][1])
best_model = resultados[best_name][0].bestModel         #  Teoría P19 (bestModel)
pred = best_model.transform(test)

### C.2 Métricas en el conjunto de prueba
Se calculan, con el mismo `MulticlassClassificationEvaluator` (cambiando `metricName`), las cuatro métricas que pide el enunciado:
- **accuracy**: proporción de aciertos totales (puede engañar con clases desbalanceadas).
- **f1**: media armónica de precisión y recall, ponderada por clase.
- **weightedPrecision**: de los que el modelo predijo como una clase, cuántos acertó, ponderado por frecuencia.
- **weightedRecall**: de los que realmente eran de una clase, cuántos detectó, ponderado por frecuencia.

 **Teoría:** **P18** (`MulticlassClassificationEvaluator`).

In [ ]:
for m in ["accuracy", "f1", "weightedPrecision", "weightedRecall"]:
    print(m, MulticlassClassificationEvaluator(labelCol="label", metricName=m).evaluate(pred))   #  Teoría P18

accuracy 0.7755102040816326
f1 0.7645189215593268
weightedPrecision 0.7691909218180456
weightedRecall 0.7755102040816326


### C.3 Matriz de confusión
Agrupa por la clase real (`label`) y la predicha (`prediction`) y cuenta. Cada fila indica cuántos estudiantes de una clase real fueron clasificados en cada clase. La diagonal (`label == prediction`) son los aciertos; fuera de la diagonal están los errores. Sirve para ver, por ejemplo, si el modelo confunde `Enrolled` con las otras clases, lo esperable al ser la clase minoritaria.

In [ ]:
pred.groupBy("label", "prediction").count().orderBy("label", "prediction").show()   # matriz de confusión

+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  0.0|       0.0|  387|
|  0.0|       1.0|    8|
|  0.0|       2.0|   23|
|  1.0|       0.0|   45|
|  1.0|       1.0|  242|
|  1.0|       2.0|   26|
|  2.0|       0.0|   75|
|  2.0|       1.0|   21|
|  2.0|       2.0|   55|
+-----+----------+-----+



### C.4 Guardado del modelo
`best_model.write().overwrite().save("modelo_final")` guarda en disco el `PipelineModel` completo (preprocesamiento + modelo). `overwrite()` permite volver a ejecutar sin error si la carpeta ya existe. La app de Streamlit lo carga después con `PipelineModel.load("modelo_final")`.

**Teoría:** **P20** (`save`).

Esa carpeta `modelo_final` debe subirse al repositorio de GitHub junto con `app.py`, `requirements.txt` y `packages.txt`, para que Streamlit Cloud pueda cargarla.

In [ ]:
best_model.write().overwrite().save("modelo_final")      #  Teoría P20 (save)

In [ ]:
#poner la carpeta modelo_final en un zip
!zip -r modelo_final.zip modelo_final

  adding: modelo_final/ (stored 0%)
  adding: modelo_final/metadata/ (stored 0%)
  adding: modelo_final/metadata/.part-00000-cd5e5810-ff8e-4815-94ad-e42f91b3d99b-c000.txt.crc (stored 0%)
  adding: modelo_final/metadata/part-00000-cd5e5810-ff8e-4815-94ad-e42f91b3d99b-c000.txt (deflated 38%)
  adding: modelo_final/metadata/._SUCCESS.crc (stored 0%)
  adding: modelo_final/metadata/_SUCCESS (stored 0%)
  adding: modelo_final/stages/ (stored 0%)
  adding: modelo_final/stages/04_StringIndexer_bb7ba63c00aa/ (stored 0%)
  adding: modelo_final/stages/04_StringIndexer_bb7ba63c00aa/metadata/ (stored 0%)
  adding: modelo_final/stages/04_StringIndexer_bb7ba63c00aa/metadata/._SUCCESS.crc (stored 0%)
  adding: modelo_final/stages/04_StringIndexer_bb7ba63c00aa/metadata/_SUCCESS (stored 0%)
  adding: modelo_final/stages/04_StringIndexer_bb7ba63c00aa/metadata/part-00000-ccafb317-d96c-450a-8dd3-d82384fe0ce9-c000.txt (deflated 40%)
  adding: modelo_final/stages/04_StringIndexer_bb7ba63c00aa/metadata/.part

In [ ]:
from google.colab import files
files.download("modelo_final.zip")

In [ ]:
import pyspark
print(pyspark.__version__)

4.0.5


## E. Mejora del modelo: EDA, ingeniería de características, selección de variables y definición del problema (2 vs. 3 clases)

**Motivación.** El modelo de las secciones A–C usa *todas* las columnas del CSV. Varias aportan poco o meten ruido:
- Códigos **nominales** (ocupación y nivel educativo de padre y madre, nacionalidad) entran como si fueran **números ordenados** (`VectorAssembler` los trata como numéricos), de modo que el modelo cree que la ocupación 9 es "mayor" que la 3.
- Hay variables **casi constantes** (por ejemplo, indicadores que casi siempre valen 0) y variables **redundantes** entre sí (matriculadas vs. evaluadas, 1.er vs. 2.º semestre).
- La clase `Enrolled` es la más difícil. En la matriz de confusión de C.3 solo se recupera el **36 %** de esos estudiantes (55 de 151).

**Reglas de este bloque (para no hacer trampa):**
1. Todo el EDA y toda la selección de variables usan **solo `train`**. `test` no se toca hasta la evaluación final (evita *data leakage*).
2. Las variables nuevas se calculan **dentro del Pipeline** (`SQLTransformer`), así el modelo guardado recibe solo las columnas crudas y la app no tiene que replicar fórmulas.
3. Toda decisión se apoya en un número o un gráfico que se genera en el propio notebook.

| Paso | Qué se hace |
|---|---|
| E.1 | EDA: calidad de datos, distribuciones, relación de cada variable con la deserción, correlaciones |
| E.2 | **Decisión: ¿3 clases o 2?** + estrategia para el desbalance |
| E.3 | Ingeniería de características (14 variables derivadas con sentido académico) |
| E.4 | Selección de variables: casi constantes → redundantes → importancia → barrido de *k* con validación cruzada |
| E.5 | Reentrenamiento en Spark ML con las variables seleccionadas (y comparación contra "todas las variables") |
| E.6 | Comparación contra el baseline, matriz de confusión, guardado y nota para la app |

### E.1 Análisis exploratorio (EDA)  *(solo sobre `train`)*

**E.1.a Preparación.**
- `train` y `test` ya traen las 3 variables derivadas de A.5, calculadas *fuera* del pipeline. Se quitan aquí y se recrean en E.3 *dentro* del pipeline.
- `NOMINAL` lista las columnas que son **códigos de categoría** (no tienen orden).
- Se pasa `train` a pandas (3 542 filas, cabe sin problema) para graficar con matplotlib/seaborn. `y = 1` si el estudiante desertó.

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
sns.set_theme(style="whitegrid")

# Derivadas de A.5 (se crearon fuera del pipeline): se quitan y se recrean en E.3 dentro del pipeline
DERIVADAS_A5 = ["tasa_aprob_s1", "tasa_aprob_s2", "delta_nota"]
raw_train = train.drop(*DERIVADAS_A5)
raw_test  = test.drop(*DERIVADAS_A5)

# Columnas que son CÓDIGOS de categoría (sin orden numérico real)
NOMINAL = ["marital_status", "application_mode", "course", "previous_qualification", "nacionality",
           "mother_s_qualification", "father_s_qualification", "mother_s_occupation", "father_s_occupation"]
COLS_EXCLUIR = ["target", "label", "label_bin", "peso", "y"]     # nunca son variables predictoras
d_idx = labels.index("Dropout")                                   # índice numérico de Dropout en la etiqueta original

pdf = raw_train.toPandas()
pdf["y"] = (pdf["target"] == "Dropout").astype(int)
print(pdf.shape, "| % deserción en train:", round(pdf["y"].mean() * 100, 1))

**E.1.b Calidad de datos y cardinalidad.** Para cada columna: nº de valores distintos, nulos, % de ceros y **% del valor dominante**. Una columna donde un solo valor supera el 98 % casi no distingue a unos estudiantes de otros (se usa como filtro en E.4). También se cuentan duplicados.

In [ ]:
X_eda = pdf.drop(columns=[c for c in COLS_EXCLUIR if c in pdf.columns])
resumen = pd.DataFrame({
    "n_unicos": X_eda.nunique(),
    "nulos": X_eda.isna().sum(),
    "ceros_%": (X_eda == 0).mean().mul(100).round(1),
    "valor_dominante_%": X_eda.apply(lambda s: s.value_counts(normalize=True).iloc[0] * 100).round(1),
})
display(resumen.sort_values("valor_dominante_%", ascending=False).head(12))
print("Filas duplicadas en train:", int(pdf.duplicated().sum()))

**E.1.c Distribución de la variable objetivo.** Izquierda: las 3 clases originales. Derecha: la versión binaria **Deserción vs. Retención** (Graduate + Enrolled). Se ve cuánto mejora el balance al pasar a 2 clases; la decisión se razona en E.2.

In [ ]:
orden3 = ["Graduate", "Dropout", "Enrolled"]
colores = {"Graduate": "#2E86C1", "Dropout": "#C0392B", "Enrolled": "#F39C12"}
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))

c3 = pdf["target"].value_counts().reindex(orden3)
ax[0].bar(c3.index, c3.values, color=[colores[k] for k in c3.index]); ax[0].set_title("3 clases (original)")
for i, v in enumerate(c3.values):
    ax[0].text(i, v, f"{v} ({v/len(pdf):.0%})", ha="center", va="bottom")

c2 = pdf["y"].map({0: "Retención", 1: "Deserción"}).value_counts().reindex(["Retención", "Deserción"])
ax[1].bar(c2.index, c2.values, color=["#2E86C1", "#C0392B"]); ax[1].set_title("2 clases (propuesta)")
for i, v in enumerate(c2.values):
    ax[1].text(i, v, f"{v} ({v/len(pdf):.0%})", ha="center", va="bottom")
plt.tight_layout(); plt.show()

**E.1.d Variables académicas según el resultado final.** *Boxplots* por clase. Lo que se busca: (1) qué variables separan a `Dropout` de las otras dos, y (2) si `Graduate` y `Enrolled` se parecen entre sí. Si se parecen, separarlos es difícil y poco útil para detectar deserción (argumento de E.2).

In [ ]:
cols_acad = ["curricular_units_1st_sem_approved", "curricular_units_1st_sem_grade",
             "curricular_units_2nd_sem_approved", "curricular_units_2nd_sem_grade",
             "curricular_units_1st_sem_enrolled", "curricular_units_2nd_sem_enrolled",
             "curricular_units_2nd_sem_without_evaluations", "age_at_enrollment"]
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, c in zip(axes.ravel(), cols_acad):
    sns.boxplot(data=pdf, x="target", y=c, order=orden3, hue="target", palette=colores, legend=False, ax=ax)
    ax.set_title(c.replace("curricular_units_", "cu_"), fontsize=10); ax.set_xlabel("")
plt.tight_layout(); plt.show()

**E.1.e Tasa de deserción por variable binaria.** Cada barra es el % de estudiantes que desertan dentro de cada valor de la variable; la línea gris es la tasa global. Una variable útil muestra barras claramente distintas entre sí y de la línea; una barra con muy pocos casos (`n` pequeño) no es confiable.

In [ ]:
cols_bin = ["tuition_fees_up_to_date", "debtor", "scholarship_holder", "gender",
            "displaced", "daytime_evening_attendance", "international", "educational_special_needs"]
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, c in zip(axes.ravel(), cols_bin):
    g = pdf.groupby(c)["y"].agg(["mean", "size"])
    ax.bar(g.index.astype(str), g["mean"] * 100, color="#C0392B")
    for i, (m, n) in enumerate(zip(g["mean"], g["size"])):
        ax.text(i, m * 100, f"n={n}", ha="center", va="bottom", fontsize=8)
    ax.axhline(pdf["y"].mean() * 100, ls="--", c="gray")
    ax.set_title(c, fontsize=10); ax.set_ylabel("% deserción")
plt.tight_layout(); plt.show()

**E.1.f Variables nominales de alta cardinalidad.** Tasa de deserción por `course` y por `application_mode` (solo categorías con al menos 30 estudiantes). Si la tasa cambia mucho entre categorías, la variable aporta y hay que codificarla como categórica (OneHot). Si todas las barras son parecidas, es ruido.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4.5))
for ax, c in zip(axes, ["course", "application_mode"]):
    g = pdf.groupby(c)["y"].agg(["mean", "size"]).query("size >= 30").sort_values("mean")
    ax.bar(g.index.astype(str), g["mean"] * 100, color="#C0392B")
    ax.axhline(pdf["y"].mean() * 100, ls="--", c="gray")
    ax.set_title(f"% deserción por {c}  (categorías con n ≥ 30)"); ax.set_xlabel(c); ax.set_ylabel("% deserción")
plt.tight_layout(); plt.show()

**E.1.g Correlaciones.**
- **Mapa de calor (Spearman)** entre las variables numéricas: bloques rojos intensos = información repetida. Se lista cada par con |ρ| ≥ 0.85.
- **Correlación con la deserción (`y`)**: ranking inicial de qué variables numéricas se asocian más con el resultado. Spearman es adecuada porque casi todas son conteos u ordinales.

In [ ]:
num_cols_eda = [c for c in X_eda.columns if c not in NOMINAL]
corr_eda = pdf[num_cols_eda].corr(method="spearman")
plt.figure(figsize=(13, 10))
sns.heatmap(corr_eda, cmap="coolwarm", center=0, vmin=-1, vmax=1, xticklabels=True, yticklabels=True)
plt.title("Correlación de Spearman entre variables numéricas"); plt.tight_layout(); plt.show()

tri = corr_eda.where(np.triu(np.ones(corr_eda.shape), k=1).astype(bool)).stack()
pares_altos = tri[tri.abs() >= 0.85].sort_values(key=abs, ascending=False)
print("Pares con |rho| >= 0.85:"); display(pares_altos.round(3).to_frame("rho"))

corr_y = pdf[num_cols_eda].corrwith(pdf["y"], method="spearman").sort_values()
fig, ax = plt.subplots(figsize=(8, 8))
corr_y.plot.barh(color=["#C0392B" if v > 0 else "#2E86C1" for v in corr_y], ax=ax)
ax.set_title("Correlación (Spearman) de cada variable con la deserción"); plt.tight_layout(); plt.show()

**Cómo se usa lo observado en E.1 (se verifica con los gráficos anteriores):**
- Lo que más separa a `Dropout` del resto debería estar en el **rendimiento académico** (aprobadas y nota, sobre todo del 2.º semestre) y en la **situación financiera** (matrícula al día, deuda, beca). Si los gráficos lo confirman, las variables derivadas de E.3 deben reforzar justo esas señales.
- Las variables con un valor dominante > 98 % y las barras casi planas en E.1.e/E.1.f son candidatas a salir.
- Los pares con |ρ| alto confirman la **redundancia** que se trata formalmente en E.4.

### E.2 ¿Tres clases o dos? Decisión y estrategia para el desbalance

**Pregunta de fondo.** El objetivo es **predecir deserción para poder intervenir**. La decisión que se toma con el modelo es binaria: *¿activo apoyo para este estudiante o no?*

**Por qué `Enrolled` no encaja como clase propia**
- `Enrolled` no es un desenlace final: son estudiantes que **siguen matriculados** al cierre de la observación. En ese momento son "retenidos"; su resultado definitivo aún no se conoce.
- Separarlos de `Graduate` es lo más difícil del problema (E.1.d permite ver cuánto se parecen) y **no sirve para la decisión de intervención**.
- Evidencia en nuestro propio baseline (matriz de C.3, 882 estudiantes de prueba): de los **198 errores**, **98 (≈ 49 %)** son confusiones *Graduate ↔ Enrolled*, que ni siquiera cambian la respuesta a "¿desertará?". Además `Enrolled` es la clase minoritaria (≈ 18 % de los datos) y su recall es de solo 36 %.
- Con 2 clases, `Deserción` pasa a ≈ 32 % de los datos frente a ≈ 68 % de `Retención`: un desbalance **moderado**, mucho más manejable que 18 %.

**Opciones consideradas**

| Opción | Ventaja | Problema | Decisión |
|---|---|---|---|
| A. 3 clases (actual) | Conserva el detalle | La mitad de los errores no importa para deserción; `Enrolled` es minoritaria y ambigua | Se mantiene como baseline |
| **B. Deserción vs. Retención (Graduate + Enrolled)** | Responde exactamente a la pregunta; usa el 100 % de los datos; mejor balance | Parte de `Enrolled` podría desertar más adelante (ruido de etiqueta, inherente al dataset) | **Elegida** |
| C. Deserción vs. Graduate (se descarta `Enrolled`) | Etiquetas "limpias" | Se pierde ≈ 18 % de los datos y, sobre todo, la app se usaría con estudiantes **activos**, justo el perfil que el modelo nunca habría visto | No recomendada |

**Cuándo conservar 3 clases:** solo si además de la deserción se quisiera identificar el grupo "rezagado" (`Enrolled`) para seguimiento; eso se haría como un segundo modelo o una segunda etapa, no como el modelo principal.

**Cómo comparar con justicia.** Las métricas de 3 clases (F1 ponderado = 0.7645) y las de 2 clases **no son comparables**. Por eso el baseline se **colapsa** a binario (predicción `Dropout` vs. no `Dropout`) y se mide con las mismas métricas que el modelo nuevo: precisión, recall y F1 **de la clase Deserción**, y AUC-ROC / AUC-PR.

**Estrategia para el desbalance:** (1) **pesos por clase** (`weightCol`, inverso de la frecuencia) para que el modelo no ignore la clase minoritaria, y (2) elegir hiperparámetros con **AUC-ROC** (no depende del umbral). Se reportan además recall y precisión de `Deserción`, que son lo que importa al intervenir.

In [ ]:
from pyspark.ml.functions import vector_to_array
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score)

def metricas(y_true, y_hat, p):
    """Métricas binarias; la clase positiva es Deserción (1). p = probabilidad de deserción."""
    return {"accuracy": accuracy_score(y_true, y_hat),
            "precision_des": precision_score(y_true, y_hat, zero_division=0),
            "recall_des": recall_score(y_true, y_hat, zero_division=0),
            "f1_des": f1_score(y_true, y_hat, zero_division=0),
            "auc_roc": roc_auc_score(y_true, p),
            "auc_pr": average_precision_score(y_true, p)}

# --- 1) Dónde están los errores del baseline de 3 clases (usa `pred` de C.1, sobre test)
cm = pred.groupBy("label", "prediction").count().toPandas()
errores = cm[cm["label"] != cm["prediction"]]["count"].sum()
errores_irrelevantes = cm[(cm["label"] != cm["prediction"]) & (cm["label"] != d_idx) & (cm["prediction"] != d_idx)]["count"].sum()
print(f"Modelo baseline: {best_name} | errores en test: {errores} | "
      f"confusiones Graduate<->Enrolled: {errores_irrelevantes} ({errores_irrelevantes/errores:.0%} de los errores)")

# --- 2) Baseline colapsado a binario (Dropout vs. resto) para compararlo con justicia
pp = pred.select("label", "prediction", vector_to_array("probability")[d_idx].alias("p")).toPandas()
base_bin = metricas((pp["label"] == d_idx).astype(int), (pp["prediction"] == d_idx).astype(int), pp["p"])
print("Baseline (3 clases) colapsado a Deserción vs. resto:")
display(pd.Series(base_bin).round(3).to_frame("valor"))

# --- 3) Etiqueta binaria y pesos por clase (solo en train; el test conserva su distribución real)
def add_bin(d):
    return d.withColumn("label_bin", F.when(F.col("target") == "Dropout", 1.0).otherwise(0.0))

raw_train, raw_test = add_bin(raw_train), add_bin(raw_test)
n = raw_train.count(); n1 = raw_train.filter("label_bin = 1.0").count(); n0 = n - n1
w1, w0 = n / (2 * n1), n / (2 * n0)                      # peso = n / (nº clases * n_clase)
raw_train = raw_train.withColumn("peso", F.when(F.col("label_bin") == 1.0, w1).otherwise(w0)).cache()
raw_test = raw_test.cache()
print(f"train: {n} filas | deserción {n1} ({n1/n:.1%}) | peso Deserción={w1:.2f}, peso Retención={w0:.2f}")

### E.3 Ingeniería de características

Se definen **14 variables derivadas** con sentido académico/financiero. Se escriben como **expresiones SQL** en un diccionario (`FE_EXPRS`), que sirve a la vez para (a) explorarlas en pandas y (b) construir el `SQLTransformer` del pipeline, así hay **una sola fuente de verdad**.

| Variable | Qué mide | Por qué podría importar |
|---|---|---|
| `tasa_aprob_s1`, `tasa_aprob_s2` | aprobadas / inscritas por semestre | Rendimiento relativo a la carga |
| `tasa_aprob_total` | aprobadas / inscritas en el año | Resume el desempeño global |
| `aprobadas_total` | aprobadas 1.er + 2.º sem. | Avance absoluto |
| `delta_aprobadas` | aprobadas 2.º − 1.er sem. | Tendencia: ¿mejora o se hunde? |
| `delta_nota` | nota 2.º − nota 1.er sem. | Tendencia de la nota |
| `nota_promedio` | promedio de notas con actividad | Una sola nota robusta a semestres vacíos |
| `tasa_eval_s1`, `tasa_eval_s2` | evaluaciones presentadas / inscritas | Compromiso con la carrera |
| `tasa_sin_eval_total` | inscritas sin evaluación / inscritas | Matricula pero no se presenta (señal de abandono) |
| `acreditadas_total` | unidades homologadas en el año | Experiencia previa |
| `sin_aprobadas_s2` | 1 si no aprobó nada en el 2.º sem. | Señal binaria fuerte de riesgo |
| `riesgo_financiero` | deudor + (1 − matrícula al día) | Combina dos señales financieras (0, 1 o 2) |
| `adulto_mayor_25` | 1 si ingresó con 25 años o más | Estudiante adulto (suele trabajar) |

Las divisiones se protegen con `CASE WHEN denominador > 0 …` para evitar división por cero. **No se asume que todas sirvan**: la selección de E.4 decide cuáles se quedan.

Después se mide cuánto se relacionan las nuevas variables con la deserción, frente a las originales.

In [ ]:
U1E, U1A, U1V, U1G, U1W, U1C = [f"curricular_units_1st_sem_{s}" for s in ["enrolled", "approved", "evaluations", "grade", "without_evaluations", "credited"]]
U2E, U2A, U2V, U2G, U2W, U2C = [f"curricular_units_2nd_sem_{s}" for s in ["enrolled", "approved", "evaluations", "grade", "without_evaluations", "credited"]]

FE_EXPRS = {
    "tasa_aprob_s1":       f"CASE WHEN {U1E} > 0 THEN {U1A} / {U1E} ELSE 0.0 END",
    "tasa_aprob_s2":       f"CASE WHEN {U2E} > 0 THEN {U2A} / {U2E} ELSE 0.0 END",
    "tasa_aprob_total":    f"CASE WHEN ({U1E} + {U2E}) > 0 THEN ({U1A} + {U2A}) / ({U1E} + {U2E}) ELSE 0.0 END",
    "aprobadas_total":     f"{U1A} + {U2A}",
    "delta_aprobadas":     f"{U2A} - {U1A}",
    "delta_nota":          f"{U2G} - {U1G}",
    "nota_promedio":       f"CASE WHEN {U1G} > 0 AND {U2G} > 0 THEN ({U1G} + {U2G}) / 2 WHEN {U1G} > 0 THEN {U1G} WHEN {U2G} > 0 THEN {U2G} ELSE 0.0 END",
    "tasa_eval_s1":        f"CASE WHEN {U1E} > 0 THEN {U1V} / {U1E} ELSE 0.0 END",
    "tasa_eval_s2":        f"CASE WHEN {U2E} > 0 THEN {U2V} / {U2E} ELSE 0.0 END",
    "tasa_sin_eval_total": f"CASE WHEN ({U1E} + {U2E}) > 0 THEN ({U1W} + {U2W}) / ({U1E} + {U2E}) ELSE 0.0 END",
    "acreditadas_total":   f"{U1C} + {U2C}",
    "sin_aprobadas_s2":    f"CASE WHEN {U2A} = 0 THEN 1 ELSE 0 END",
    "riesgo_financiero":   "debtor + (1 - tuition_fees_up_to_date)",
    "adulto_mayor_25":     "CASE WHEN age_at_enrollment >= 25 THEN 1 ELSE 0 END",
}
FE_NAMES = list(FE_EXPRS)

def add_fe(d):
    """Agrega las variables derivadas a un DataFrame de Spark (solo para explorar; en el pipeline se usa SQLTransformer)."""
    return d.selectExpr("*", *[f"({e}) AS {n}" for n, e in FE_EXPRS.items()])

pdf_fe = add_fe(raw_train).toPandas()
pdf_fe["y"] = (pdf_fe["target"] == "Dropout").astype(int)

rho_nuevas = pdf_fe[FE_NAMES].corrwith(pdf_fe["y"], method="spearman")
rho_orig = pdf_fe[num_cols_eda].corrwith(pdf_fe["y"], method="spearman")
print("Las 8 variables ORIGINALES más asociadas a la deserción (|rho|):")
display(rho_orig.reindex(rho_orig.abs().sort_values(ascending=False).index).head(8).round(3).to_frame("rho"))

fig, ax = plt.subplots(figsize=(8, 5))
rho_nuevas.sort_values().plot.barh(color=["#C0392B" if v > 0 else "#2E86C1" for v in rho_nuevas.sort_values()], ax=ax)
ax.set_title("Variables NUEVAS: correlación (Spearman) con la deserción"); plt.tight_layout(); plt.show()

top_fe = rho_nuevas.abs().sort_values(ascending=False).index[:4]
fig, axes = plt.subplots(1, 4, figsize=(16, 3.8))
for ax, c in zip(axes, top_fe):
    sns.boxplot(data=pdf_fe, x="target", y=c, order=orden3, hue="target", palette=colores, legend=False, ax=ax)
    ax.set_title(c, fontsize=10); ax.set_xlabel("")
plt.tight_layout(); plt.show()

### E.4 Selección de variables

**Candidatas:** las columnas originales (menos `target`, `label`) + las 14 derivadas. Todo se calcula **solo con `train`** y contra la **deserción** (la etiqueta binaria decidida en E.2). Cuatro filtros en cadena:

1. **Casi constantes** (≥ 98 % un solo valor): no distinguen estudiantes → se descartan.
2. **Redundantes** (|ρ de Spearman| ≥ 0.90 con otra variable): de cada par se conserva la que tenga **mayor información mutua** con la deserción. Esto aplica también a las derivadas que repiten una original.
3. **Relevancia**: se ordenan por la combinación de dos medidas independientes:
   - **Información mutua** (captura relaciones no lineales).
   - **Importancia por permutación** de un Random Forest, medida en datos de validación: cuánto *empeora el AUC* si se desordena esa variable. Se prefiere a la importancia de impureza porque esta favorece a las variables con muchos valores (justo el caso de los códigos de ocupación y nivel educativo).
4. **Barrido de *k* con validación cruzada (5 folds)**: se entrena con las *k* mejores variables para varios *k* y se elige el **menor *k*** cuyo AUC queda dentro de **una desviación estándar** del mejor (regla de "1 error estándar": la solución más simple que no pierde rendimiento).

*Limitación honesta:* las medidas de relevancia miden cada variable en presencia de las demás y con códigos nominales tratados como números (para el ranking). El modelo final sí los codifica como categóricos (OneHot). Por eso el paso 4 y el experimento de E.5 son la validación real.

In [ ]:
from sklearn.feature_selection import mutual_info_classif
from sklearn.ensemble import RandomForestClassifier as SkRF
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score

cand = [c for c in pdf_fe.columns if c not in COLS_EXCLUIR]
X = pdf_fe[cand].astype(float)
y = pdf_fe["y"].values
print(f"{len(cand)} variables candidatas ({len(cand) - len(FE_NAMES)} originales + {len(FE_NAMES)} derivadas)")

# 1) Casi constantes
top_share = X.apply(lambda s: s.value_counts(normalize=True).iloc[0])
quasi_const = top_share[top_share >= 0.98].index.tolist()
print("\n1) Casi constantes (>=98% un solo valor):", quasi_const)

# Información mutua con la deserción (para decidir entre variables redundantes y para el ranking)
discretas = [bool((X[c] % 1 == 0).all() and X[c].nunique() <= 30) for c in cand]
mi = pd.Series(mutual_info_classif(X, y, discrete_features=discretas, random_state=42), index=cand)

# 2) Redundantes: de cada par muy correlacionado se queda la de mayor información mutua
corr = X.drop(columns=quasi_const).corr(method="spearman").abs()
cols_c = list(corr.columns)
red_reason = {}                                  # variable descartada -> variable que se conserva
for i, a in enumerate(cols_c):
    for b in cols_c[i + 1:]:
        if a in red_reason or b in red_reason:
            continue
        if corr.loc[a, b] >= 0.90:
            perdedora, ganadora = (a, b) if mi[a] < mi[b] else (b, a)
            red_reason[perdedora] = ganadora
print("\n2) Redundantes (|rho| >= 0.90) -> descartada: conserva")
for k, v in red_reason.items():
    print(f"   {k}  ->  {v}")

In [ ]:
# 3) Relevancia: información mutua + importancia por permutación (AUC) en datos de validación
cand2 = [c for c in cand if c not in quasi_const and c not in red_reason]
Xtr, Xva, ytr, yva = train_test_split(X[cand2], y, test_size=0.25, stratify=y, random_state=42)
rf_sk = SkRF(n_estimators=300, min_samples_leaf=3, class_weight="balanced", n_jobs=-1, random_state=42).fit(Xtr, ytr)
pi = permutation_importance(rf_sk, Xva, yva, scoring="roc_auc", n_repeats=15, random_state=42, n_jobs=-1)

tabla = pd.DataFrame({"mi": mi[cand2], "perm_mean": pi.importances_mean, "perm_std": pi.importances_std}, index=cand2)
tabla["score_rank"] = (tabla["mi"].rank(ascending=False) + tabla["perm_mean"].rank(ascending=False)) / 2
tabla = tabla.sort_values("score_rank")
ranking = tabla.index.tolist()

top = tabla.head(25).iloc[::-1]
fig, axes = plt.subplots(1, 2, figsize=(15, 7))
axes[0].barh(top.index, top["perm_mean"], xerr=top["perm_std"], color="#C0392B")
axes[0].set_title("Importancia por permutación (caída de AUC)")
axes[1].barh(top.index, top["mi"], color="#2E86C1")
axes[1].set_title("Información mutua con la deserción")
plt.tight_layout(); plt.show()
display(tabla.head(15).round(4))

In [ ]:
# 4) Barrido de k con validación cruzada (solo train). Referencia: todas las variables candidatas
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
def cv_auc(cols):
    m = SkRF(n_estimators=200, min_samples_leaf=3, class_weight="balanced", n_jobs=-1, random_state=42)
    s = cross_val_score(m, X[cols], y, cv=skf, scoring="roc_auc", n_jobs=1)
    return s.mean(), s.std()

ks = sorted({k for k in [5, 8, 10, 12, 15, 20, 25, len(ranking)] if k <= len(ranking)})
barrido = pd.DataFrame([(k, *cv_auc(ranking[:k])) for k in ks], columns=["k", "auc_cv", "std"])
auc_todas, std_todas = cv_auc(cand)

mejor = barrido.loc[barrido["auc_cv"].idxmax()]
umbral = mejor["auc_cv"] - mejor["std"]                      # regla de 1 error estándar
k_opt = int(barrido[barrido["auc_cv"] >= umbral]["k"].min())
display(barrido.round(4))
print(f"AUC CV con TODAS las candidatas ({len(cand)}): {auc_todas:.4f} ± {std_todas:.4f}")
print(f"Mejor k: {int(mejor['k'])} (AUC {mejor['auc_cv']:.4f}) | k elegido (1 error estándar): {k_opt}")

plt.figure(figsize=(8, 4))
plt.errorbar(barrido["k"], barrido["auc_cv"], yerr=barrido["std"], marker="o", color="#C0392B", capsize=3, label="Top-k variables")
plt.axhline(auc_todas, ls="--", c="gray", label=f"Todas ({len(cand)})")
plt.axvline(k_opt, ls=":", c="#2E86C1", label=f"k elegido = {k_opt}")
plt.xlabel("k (nº de variables)"); plt.ylabel("AUC-ROC (CV 5 folds)"); plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Lista final y razón de cada decisión
sel_features = ranking[:k_opt]
sel_fe  = [c for c in sel_features if c in FE_NAMES]
sel_cat = [c for c in sel_features if c in NOMINAL and X[c].nunique() > 2]     # nominales -> OneHot
sel_num = [c for c in sel_features if c not in sel_cat]                          # resto va como numérica

estado = {}
for c in cand:
    if c in quasi_const:      estado[c] = f"descartada: casi constante ({top_share[c]:.1%} un solo valor)"
    elif c in red_reason:     estado[c] = f"descartada: redundante con {red_reason[c]}"
    elif c in sel_features:   estado[c] = "SELECCIONADA"
    else:                     estado[c] = "descartada: aporte bajo (fuera del top-k)"
estado = pd.Series(estado, name="decisión").to_frame()
estado["tipo"] = ["derivada" if c in FE_NAMES else "original" for c in estado.index]
display(estado.sort_values("decisión"))

print(f"\nSeleccionadas: {len(sel_features)} de {len(cand)}")
print("  numéricas :", sel_num)
print("  categóricas (OneHot):", sel_cat)
print("  derivadas que sobreviven:", sel_fe)

### E.5 Reentrenamiento en Spark ML con las variables seleccionadas

Se repite el esquema de B (Pipeline + CrossValidator con 3 folds), con tres cambios:

1. **Etiqueta binaria** `label_bin` y **pesos por clase** (`weightCol="peso"`).
2. El pipeline empieza con un **`SQLTransformer`** que crea las variables derivadas, y solo usa las columnas **seleccionadas** (numéricas directas + categóricas con OneHot).
3. Se agrega **`GBTClassifier`** (árboles con *boosting*), que en Spark solo soporta clasificación binaria, otra ventaja de pasar a 2 clases. Regresión logística usa `family="binomial"`.

Los hiperparámetros se eligen con **AUC-ROC** (`BinaryClassificationEvaluator`).

**Experimento de control (`rf_todas`).** Un Random Forest con **las mismas condiciones** (etiqueta binaria, pesos, malla de hiperparámetros, mismo split) pero con **todas las variables del baseline** (`num_cols` + `cat_cols` de B.1 + las 3 derivadas de A.5). Compararlo con `rf_sel` aísla el efecto de la selección de variables: cualquier diferencia se debe solo a qué columnas entran.

In [ ]:
from pyspark.ml.feature import SQLTransformer
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
import time

def build_pipe(est, num, cat, fe_names=()):
    """Pipeline autosuficiente: derivadas (SQL) -> StringIndexer -> OneHot -> VectorAssembler -> StandardScaler -> modelo."""
    stages = []
    if fe_names:
        sql = "SELECT *, " + ", ".join(f"({FE_EXPRS[n]}) AS {n}" for n in fe_names) + " FROM __THIS__"
        stages.append(SQLTransformer(statement=sql))
    stages += [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in cat]
    entradas = list(num)
    if cat:
        stages.append(OneHotEncoder(inputCols=[f"{c}_idx" for c in cat], outputCols=[f"{c}_ohe" for c in cat], handleInvalid="keep"))
        entradas += [f"{c}_ohe" for c in cat]
    stages += [VectorAssembler(inputCols=entradas, outputCol="features_raw", handleInvalid="keep"),
               StandardScaler(inputCol="features_raw", outputCol="features", withMean=False, withStd=True),
               est]
    return Pipeline(stages=stages)

def est_y_malla(tipo):
    kw = dict(labelCol="label_bin", featuresCol="features", weightCol="peso")
    if tipo == "rf":
        est = RandomForestClassifier(seed=42, **kw)
        grid = ParamGridBuilder().addGrid(est.numTrees, [100, 200]).addGrid(est.maxDepth, [8, 12]).build()
    elif tipo == "lr":
        est = LogisticRegression(family="binomial", maxIter=100, **kw)
        grid = ParamGridBuilder().addGrid(est.regParam, [0.0, 0.01, 0.1]).addGrid(est.elasticNetParam, [0.0, 0.5]).build()
    else:
        est = GBTClassifier(seed=42, **kw)
        grid = ParamGridBuilder().addGrid(est.maxDepth, [3, 5]).addGrid(est.maxIter, [50, 100]).build()
    return est, grid

def eval_bin(pred_df):
    pp = pred_df.select("label_bin", "prediction", vector_to_array("probability")[1].alias("p")).toPandas()
    return metricas(pp["label_bin"].astype(int), pp["prediction"].astype(int), pp["p"])

# Variables del baseline (para el experimento de control)
base_num = [c for c in raw_train.columns if c not in cat_cols + COLS_EXCLUIR] + DERIVADAS_A5
configs = {                     # nombre: (tipo de modelo, numéricas, categóricas, derivadas a crear en el pipeline)
    "rf_todas": ("rf",  base_num, cat_cols, DERIVADAS_A5),
    "rf_sel":   ("rf",  sel_num,  sel_cat,  sel_fe),
    "lr_sel":   ("lr",  sel_num,  sel_cat,  sel_fe),
    "gbt_sel":  ("gbt", sel_num,  sel_cat,  sel_fe),
}
ev_cv = BinaryClassificationEvaluator(labelCol="label_bin", rawPredictionCol="rawPrediction", metricName="areaUnderROC")

resultados_bin = {}
for i, (nombre, (tipo, num, cat, fe)) in enumerate(configs.items(), start=1):
    est, grid = est_y_malla(tipo)
    print(f"[{i}/{len(configs)}] {nombre}: {len(num)} numéricas + {len(cat)} categóricas | {len(grid)} combinaciones x 3 folds", flush=True)
    t0 = time.time()
    cvm = CrossValidator(estimator=build_pipe(est, num, cat, fe), estimatorParamMaps=grid,
                         evaluator=ev_cv, numFolds=3, seed=42).fit(raw_train)
    resultados_bin[nombre] = {"modelo": cvm, "cv_auc": max(cvm.avgMetrics), "test": eval_bin(cvm.transform(raw_test))}
    print(f"     AUC CV = {resultados_bin[nombre]['cv_auc']:.4f} | {time.time() - t0:.0f} s", flush=True)

### E.6 Resultados, comparación con el baseline y guardado

- La **tabla** pone en la misma escala el baseline (3 clases colapsado a Deserción vs. resto, E.2) y los modelos nuevos, evaluados sobre el **mismo `test`**.
- El **mejor modelo se elige por AUC de validación cruzada** (calculado solo con `train`), **no por el resultado en `test`**. Así `test` sigue siendo una medida honesta.
- Cómo leerla: `recall_des` = de los que realmente desertan, cuántos detecta (importante para no dejar a nadie sin apoyo); `precision_des` = de los que marca como desertores, cuántos lo son (importante para no gastar recursos de más).
- La comparación `rf_todas` vs. `rf_sel` muestra el efecto **puro** de la selección de variables.

In [ ]:
filas = {"baseline (3 clases → binario)": {"cv_auc": np.nan, **base_bin}}
for nombre, r in resultados_bin.items():
    filas[nombre] = {"cv_auc": r["cv_auc"], **r["test"]}
tabla_res = pd.DataFrame(filas).T
display(tabla_res.round(3))

# Mejor modelo con variables seleccionadas, elegido por AUC de CV
best_bin_name = max([k for k in resultados_bin if k.endswith("_sel")], key=lambda k: resultados_bin[k]["cv_auc"])
best_bin = resultados_bin[best_bin_name]["modelo"].bestModel
print("Mejor modelo (por AUC de CV):", best_bin_name)
print("Hiperparámetros ganadores:", {p.name: v for p, v in best_bin.stages[-1].extractParamMap().items()
                                    if p.name in ("numTrees", "maxDepth", "maxIter", "regParam", "elasticNetParam")})

# Matriz de confusión binaria sobre test
pred_bin = best_bin.transform(raw_test)
mc = pred_bin.groupBy("label_bin", "prediction").count().toPandas()
mc["real"] = mc["label_bin"].map({0.0: "Retención", 1.0: "Deserción"})
mc["predicho"] = mc["prediction"].map({0.0: "Retención", 1.0: "Deserción"})
display(mc.pivot(index="real", columns="predicho", values="count").fillna(0).astype(int))

In [ ]:
# Guardado del mejor modelo binario (PipelineModel completo: derivadas + preprocesamiento + clasificador)
best_bin.write().overwrite().save("modelo_final_binario")

# Columnas CRUDAS que la app debe enviar (las derivadas las calcula el propio pipeline)
cols_crudas = [c for c in raw_train.columns if c not in COLS_EXCLUIR]
usadas = {c for c in sel_features if c in cols_crudas}
for n in sel_fe:
    usadas |= {c for c in cols_crudas if re.search(rf"\b{c}\b", FE_EXPRS[n])}
print(f"La app debe enviar {len(usadas)} columnas crudas:\n", sorted(usadas))
print('\nEn la app: LABELS = ["Retención", "Deserción"]  (índice 0 = Retención, 1 = Deserción)')
print("Riesgo de deserción = probability[1] de la predicción del pipeline.")

In [ ]:
# Comprimir y descargar la carpeta del modelo para subirla a GitHub junto con app.py
import shutil
from google.colab import files
shutil.make_archive("modelo_final_binario", "zip", ".", "modelo_final_binario")
files.download("modelo_final_binario.zip")

In [ ]:
# Resumen automático (se actualiza con los resultados de esta ejecución)
b, a = tabla_res.loc["baseline (3 clases → binario)"], tabla_res.loc[best_bin_name]
t, s = tabla_res.loc["rf_todas"], tabla_res.loc["rf_sel"]
n_base = len(base_num) + len(cat_cols)
print(f"Variables: {n_base} en el baseline → {len(sel_features)} seleccionadas ({len(sel_fe)} derivadas).")
print(f"Efecto de la selección (rf_todas → rf_sel): AUC {t.auc_roc:.3f} → {s.auc_roc:.3f} | F1 Deserción {t.f1_des:.3f} → {s.f1_des:.3f}")
print(f"Baseline colapsado → {best_bin_name}: AUC {b.auc_roc:.3f} → {a.auc_roc:.3f} | "
      f"recall Deserción {b.recall_des:.3f} → {a.recall_des:.3f} | precisión {b.precision_des:.3f} → {a.precision_des:.3f} | "
      f"F1 {b.f1_des:.3f} → {a.f1_des:.3f}")

**Cómo interpretar el cierre (con los números de la tabla E.6):**
- Si `rf_sel` ≈ `rf_todas` (o mejor) con **menos variables**, la selección cumplió su objetivo: un modelo más simple, más rápido y más fácil de explicar, sin perder rendimiento.
- Si el modelo nuevo mejora el **AUC** y el **F1 de Deserción** frente al baseline colapsado, el replanteamiento (2 clases + variables útiles + pesos) fue positivo. Si la mejora es pequeña, también es un resultado válido: significa que el baseline ya capturaba casi toda la señal disponible y el beneficio principal es la **simplicidad y la alineación con el objetivo**.
- Un modelo con recall alto y precisión más baja detecta a casi todos los desertores a costa de falsas alarmas; el umbral de decisión (0.5 por defecto) se puede mover según el costo de cada error.

##  Resumen: cobertura de la rúbrica y pendientes

| Criterio (peso) | Estado en este notebook | Dónde |
|---|---|---|
| Carga y exploración (10 %) | ✔️ SparkSession, esquema, dimensiones, distribución de clases. Análisis del desbalance y EDA completo en **E.1–E.2** | A.2, A.4 |
| Preprocesamiento (10 %) | ✔️ nombres normalizados, duplicados, nulos, partición 80/20 con semilla 42.  Estrategia de desbalance (pesos por clase + decisión 2 vs. 3 clases) en **E.2** | A.3, A.6 |
| Ingeniería de características (15 %) | ✔️ 3 variables derivadas, codificación, ensamblado y escalado en el pipeline.  Selección de variables justificada (información mutua, permutación, barrido de k) en **E.3–E.4** | A.5, B.2 |
| Pipelines y modelos (15 %) | ✔️ 3 modelos, cada uno en un Pipeline completo | B.3, B.6 |
| Hiperparametrización (15 %) | ✔️ 2 hiperparámetros por modelo, CrossValidator con 3 folds.  Falta **analizar los mejores parámetros** | B.4, B.6 |
| Evaluación y selección (10 %) | ✔️ 4 métricas, matriz de confusión y guardado.  Tabla comparativa baseline vs. modelos mejorados en **E.6** | C.1 – C.4 |
| App Streamlit: funcionalidad (15 %) | Carga del modelo lista; falta el formulario completo, la predicción y las probabilidades, y el despliegue | D.3, D.5 |
| App Streamlit: diseño (10 %) | ⚠️ Título, banner, instrucciones y estilo listos; falta el **autor al final** | D.2, D.4 |

**Tip de estudio:** para repasar la parte teórica, busque en el notebook el texto ` Teoría P` y verá, en orden, las 20 respuestas en contexto de código real.